# Notebook 05 — Claude Annotator Benchmark on Gold Test Set

**Purpose:** Evaluate Claude's zero/few-shot accuracy on the 1,700-sample gold test set
using the *identical* few-shot prompt used in Phase 3 denoising. This benchmarks whether
Claude was a qualified annotator for Bahasa Rojak before being used to relabel training data.

**Addresses reviewer feedback:** "Before unleashing an LLM to act as an oracle... you must
evaluate the LLM's zero-shot or few-shot accuracy on a subset of your clean gold test set."

**Cost:** ~$1.00 API cost | **Runtime:** ~15–20 minutes

In [1]:
import sys
sys.path.insert(0, r'D:\Doc\Programming in AI\Project\src')  # absolute path -- works regardless of Jupyter launch dir

import anthropic, json, time, random, os, re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, f1_score

from config import RESULTS_DIR, CLEAN_TEST_CSV, LABEL2ID, ID2LABEL, SEED, LLM_CONFIG

CLAUDE_MODEL    = 'claude-sonnet-4-6'
BATCH_SIZE      = 10        # 170 batches for 1700 samples
REQUEST_PAUSE   = 1.2       # seconds between calls (Claude: 50 RPM limit)
LLM_BENCH_DIR   = RESULTS_DIR / 'llm_benchmark'
LLM_BENCH_DIR.mkdir(exist_ok=True)

print(f'Model: {CLAUDE_MODEL}')
print(f'Test set: {CLEAN_TEST_CSV}')
print(f'Output: {LLM_BENCH_DIR}')

Model: claude-sonnet-4-6
Test set: D:\Doc\Programming in AI\Project\results\test_clean_test_split.csv
Output: D:\Doc\Programming in AI\Project\results\llm_benchmark


In [2]:
import os
# os.environ['ANTHROPIC_API_KEY'] = ''  # REMOVED: set ANTHROPIC_API_KEY as env var before running
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
print('Client ready.')

Client ready.


## Load Data & Reconstruct Few-Shot Prompt

Reuses the 30 few-shot examples saved during Phase 3 (prompt_dev_set.csv).
This ensures the benchmark uses the *identical* prompt Claude saw during denoising.

In [3]:
def preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load few-shot examples from Phase 3
few_shot_path = RESULTS_DIR / 'phase3_v2' / 'prompt_dev_set.csv'
df_few_shot   = pd.read_csv(few_shot_path)

# Load gold test set (1,700 samples)
df_test = pd.read_csv(CLEAN_TEST_CSV)
df_test['text_clean'] = df_test['text'].apply(preprocess)

# Reconstruct few-shot block (same format as Phase 3)
few_shot_lines = []
for label in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
    few_shot_lines.append(f'--- {label} examples ---')
    examples = df_few_shot[df_few_shot['label'] == label]
    for _, row in examples.iterrows():
        few_shot_lines.append(f'Tweet: {row["text_clean"]}' )
        few_shot_lines.append(f'Label: {label}')
        few_shot_lines.append('')
FEW_SHOT_BLOCK = '\n'.join(few_shot_lines)

SYSTEM_PROMPT = (
    "You are an expert annotator for Malay-English code-switched (Bahasa Rojak) "
    "social media sentiment analysis.\n\n"
    "## Malaysian Twitter conventions\n"
    '- "lah", "leh", "lor", "kan", "je" are Malay discourse particles — tone modifiers, not sentiment\n'
    '- "tak" = "tidak" = negation ("tak best" = not good)\n'
    '- "best" = good/enjoyable in Malay slang\n'
    "- Hashtags and @ mentions may carry sentiment signals\n"
    '- Informal spelling and repeated letters amplify sentiment\n\n'
    "## Label definitions\n"
    "- POSITIVE: happiness, praise, approval, excitement, or satisfaction\n"
    "- NEGATIVE: sadness, complaint, anger, criticism, disappointment, or frustration\n"
    "- NEUTRAL: factual statement, question, announcement, or mixed/no clear polarity\n\n"
    "## Few-shot examples (high-confidence, unambiguous)\n\n"
    + FEW_SHOT_BLOCK
)

print(f'Few-shot examples loaded: {len(df_few_shot)}')
print(f'Test samples: {len(df_test)}')
print(f'System prompt length: {len(SYSTEM_PROMPT):,} chars')

Few-shot examples loaded: 15
Test samples: 1700
System prompt length: 2,592 chars


## Batch API Calls

In [4]:
def call_batch(tweets: list, max_retries: int = 6) -> list:
    """Send a batch of tweets to Claude, return list of predicted labels."""
    numbered = '\n'.join(f'{i+1}. {t}' for i, t in enumerate(tweets))
    user_msg = (
        f"Classify each of the {len(tweets)} tweets below.\n"
        f"Return ONLY a JSON array with one object per tweet in order:\n"
        f'[{{"id": 1, "label": "POSITIVE"}}, {{"id": 2, "label": "NEUTRAL"}}, ...]\n\n'
        f"Tweets:\n{numbered}"
    )
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=300,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}]
            )
            raw = resp.content[0].text.strip()
            raw = re.sub(r'^```(?:json)?\s*', '', raw)
            raw = re.sub(r'\s*```$', '', raw)
            parsed = json.loads(raw)
            labels = [item['label'].upper().strip() for item in parsed]
            assert len(labels) == len(tweets), f'Expected {len(tweets)} labels, got {len(labels)}'
            assert all(l in ('POSITIVE', 'NEGATIVE', 'NEUTRAL') for l in labels), f'Invalid label in {labels}'
            return labels
        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f'  Retry {attempt+1}/{max_retries}: {type(e).__name__}: {e}. Waiting {wait:.1f}s')
            time.sleep(wait)
    print(f'  FAILED after {max_retries} retries — falling back to NEUTRAL for this batch')
    return ['NEUTRAL'] * len(tweets)

## Run Benchmark (with checkpoint/resume support)

In [5]:
checkpoint_path = LLM_BENCH_DIR / 'benchmark_checkpoint.csv'

# Resume if interrupted
if checkpoint_path.exists():
    df_done   = pd.read_csv(checkpoint_path)
    done_texts = set(df_done['text_clean'].tolist())
    print(f'Resuming: {len(df_done)}/{len(df_test)} done')
else:
    df_done    = pd.DataFrame(columns=['text_clean', 'true_label', 'pred_label'])
    done_texts = set()

df_remaining = df_test[~df_test['text_clean'].isin(done_texts)].reset_index(drop=True)
results = df_done.to_dict('records')

n_remaining = len(df_remaining)
n_batches   = (n_remaining + BATCH_SIZE - 1) // BATCH_SIZE
print(f'Remaining: {n_remaining} samples in {n_batches} batches')

for batch_idx, batch_start in enumerate(range(0, n_remaining, BATCH_SIZE)):
    batch  = df_remaining.iloc[batch_start : batch_start + BATCH_SIZE]
    tweets = batch['text_clean'].tolist()
    preds  = call_batch(tweets)

    for row, pred in zip(batch.itertuples(), preds):
        results.append({
            'text_clean': row.text_clean,
            'true_label': row.label,
            'pred_label': pred,
        })

    time.sleep(REQUEST_PAUSE)

    # Save checkpoint every 10 batches
    if (batch_idx + 1) % 10 == 0:
        pd.DataFrame(results).to_csv(checkpoint_path, index=False)
    done_so_far = batch_start + len(tweets)
    print(f'  {done_so_far + len(df_done)}/{len(df_test)} ({(done_so_far/n_remaining)*100:.0f}%)', end='\r')

df_results = pd.DataFrame(results)
df_results.to_csv(checkpoint_path, index=False)
print(f'\nBenchmark complete: {len(df_results)} samples')

Remaining: 1700 samples in 170 batches
  Retry 1/6: JSONDecodeError: Expecting value: line 1 column 1 (char 0). Waiting 1.4s
  Retry 2/6: JSONDecodeError: Expecting value: line 1 column 1 (char 0). Waiting 2.2s
  Retry 1/6: JSONDecodeError: Expecting value: line 1 column 1 (char 0). Waiting 1.9s
  Retry 2/6: JSONDecodeError: Expecting value: line 1 column 1 (char 0). Waiting 2.9s
  Retry 3/6: JSONDecodeError: Expecting value: line 1 column 1 (char 0). Waiting 4.2s
  Retry 4/6: JSONDecodeError: Expecting value: line 1 column 1 (char 0). Waiting 8.0s
  Retry 5/6: JSONDecodeError: Expecting value: line 1 column 1 (char 0). Waiting 16.1s
  Retry 6/6: JSONDecodeError: Expecting value: line 1 column 1 (char 0). Waiting 32.8s
  FAILED after 6 retries — falling back to NEUTRAL for this batch
  1700/1700 (100%)
Benchmark complete: 1700 samples


## Results & Comparison

In [6]:
true_labels = df_results['true_label'].tolist()
pred_labels = df_results['pred_label'].tolist()
label_order = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']

macro_f1 = f1_score(true_labels, pred_labels, average='macro', labels=label_order)
report   = classification_report(
    true_labels, pred_labels,
    labels=label_order, target_names=label_order,
    output_dict=True,
)

print(f'=== Claude {CLAUDE_MODEL} — Few-Shot Benchmark (n={len(df_results)}) ===')
print(f'Macro-F1: {macro_f1:.4f}')
print()
print(classification_report(true_labels, pred_labels, labels=label_order, target_names=label_order))

# Comparison table
print('=== Comparison: Claude vs Fine-tuned Models ===')
print(f'{"Model":<40} {"Macro-F1":>10}  {"Notes"}')
print('-' * 75)
rows = [
    ('TF-IDF + SVM',                 0.6407, 'Classical ML baseline'),
    ('XLM-R v2 (fine-tuned)',         0.6587, 'Transformer baseline'),
    ('XLM-R Focal γ=1.0 (best)',      0.6841, 'Best fine-tuned model'),
    ('XLM-R v2 Denoised (Phase 3)',   0.6305, 'LLM-denoised training set'),
    (f'Claude {CLAUDE_MODEL} (few-shot)', macro_f1, '30 in-context examples, same prompt as Phase 3'),
]
for name, f1, note in rows:
    marker = ' ← THIS' if 'Claude' in name else ''
    print(f'{name:<40} {f1:>10.4f}  {note}{marker}')

print()
if macro_f1 < 0.6587:
    print(f'INTERPRETATION: Claude ({macro_f1:.4f}) scores BELOW the XLM-R v2 baseline (0.6587).')
    print('This confirms Claude was not a superior annotator for Bahasa Rojak,')
    print('explaining why Phase 3 denoising produced a null result.')
elif macro_f1 < 0.6841:
    print(f'INTERPRETATION: Claude ({macro_f1:.4f}) is competitive but below the best fine-tuned model.')
    print('LLM annotations may still introduce noise on ambiguous high-entropy samples.')
else:
    print(f'INTERPRETATION: Claude ({macro_f1:.4f}) matches or exceeds fine-tuned models.')
    print('The Phase 3 null result may reflect distributional shift from denoising high-entropy samples.')

# Save
bench_results = {
    'model':     CLAUDE_MODEL,
    'eval_type': 'few-shot (30 examples — identical to Phase 3 denoising prompt)',
    'test_n':    len(df_results),
    'macro_f1':  round(macro_f1, 4),
    'per_class': {
        cls: {k: round(v, 4) for k, v in report[cls].items()
              if k in ('precision', 'recall', 'f1-score')}
        for cls in label_order
    },
    'note': (
        'Benchmarks Claude annotator quality on the gold test set. '
        f'If Macro-F1 < 0.6587 (XLM-R v2 baseline), Claude was not a superior '
        'annotator for this dialect, providing a mechanistic explanation for '
        'the Phase 3 denoising null result.'
    ),
}
with open(LLM_BENCH_DIR / 'benchmark_results.json', 'w') as f:
    json.dump(bench_results, f, indent=2)
print(f'\nSaved → {LLM_BENCH_DIR / "benchmark_results.json"}')

=== Claude claude-sonnet-4-6 — Few-Shot Benchmark (n=1700) ===
Macro-F1: 0.6000

              precision    recall  f1-score   support

    NEGATIVE       0.54      0.58      0.56       330
     NEUTRAL       0.86      0.63      0.73      1178
    POSITIVE       0.36      0.91      0.51       192

    accuracy                           0.65      1700
   macro avg       0.59      0.70      0.60      1700
weighted avg       0.74      0.65      0.67      1700

=== Comparison: Claude vs Fine-tuned Models ===
Model                                      Macro-F1  Notes
---------------------------------------------------------------------------
TF-IDF + SVM                                 0.6407  Classical ML baseline
XLM-R v2 (fine-tuned)                        0.6587  Transformer baseline
XLM-R Focal γ=1.0 (best)                     0.6841  Best fine-tuned model
XLM-R v2 Denoised (Phase 3)                  0.6305  LLM-denoised training set
Claude claude-sonnet-4-6 (few-shot)          0.6000 